In [1]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_DimGroup V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/Dim_Group" # ← Change source path
TARGET_PATH = "abfss://Gold/Dim_Group" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 3, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_DimGroup V2...
🚀 Starting ntk_Sil2Gld_DimGroup V2


In [5]:
# Section --- Importing modules and Defining Variable for standard usage source / target.
from pyspark.sql import SparkSession
from datetime import datetime
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

# Get today's date and format it
today = datetime.now()  
from datetime import datetime, timedelta
###today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")

# Build dynamic source folder structure
sourcefolder_structure = f"Silver_layer/Reporting/{year}/{month}/{day}"   
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "Dim_Group.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"/PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# Filename
target_filename = "Dim_Group.parquet"
print(f"Variables created and session started.")



StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 7, Finished, Available, Finished)

Variables created and session started.


In [6]:
# Initialize Spark session
spark = SparkSession.builder.appName("SilverToGold_FileReader").getOrCreate()

# Full Source  file path
full_source_path = f"{complete_source_path}/{source_filename}"

# Full Target  file path
full_target_path = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_path}")
print(f"Target: {full_target_path}")

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 8, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/06/Dim_Group.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Group.parquet


In [7]:
# # Reading source and target data to dataframes.

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col, when

# columns_needed = ["User_GUIDPK","UserPrincipalName","Email"]
# Read Parquet files directly into memory
print("Loading Parquet files into memory...")
source_df = spark.read.parquet(full_source_path)

# source_df = source_df.select(*columns_needed)
# source_df.printSchema()

target_df = spark.read.parquet(full_target_path)

# try:
#     target_df = spark.read.parquet(full_target_path)
# except:
    # # Create empty DataFrame with same schema if target doesn't exist
    # target_df = spark.createDataFrame([], source_df.schema)

# columns_needed = ["UserKey","UserPrincipalName","Email"]
# target_df = target_df.select(*columns_needed)
# target_df.printSchema()

print(f"Reading source file completed {full_source_path}")
print(f"Reading target file completed {full_target_path}")

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 9, Finished, Available, Finished)

Loading Parquet files into memory...
Reading source file completed abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/06/Dim_Group.parquet
Reading target file completed abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Group.parquet


#### Caches

In [8]:
# Cache for performance
target_df.cache()
print(target_df.count())

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 10, Finished, Available, Finished)

212


In [9]:
source_df.show(2)
target_df = target_df.withColumn("IsDeleted", lit(0))
target_df.show(0)

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 11, Finished, Available, Finished)

+--------------------+---------+--------------------+-------+---------------+--------------------+----------+-----------+----------+-------------+------------+------------------------------+--------------------------+-----------------------+----------------------------+------------------------------+----------+----------+--------------------+-----------------------+--------------------+----------------+--------------+-------------------+------------+--------------------+-------------+--------------------+--------------------+--------------------+-----------+
|            GroupKey|GroupGuid|           LoginName|GroupId|  PrincipalType|      OwnerLoginName|OwnerEmail|MemberCount| GroupType|IsSystemGroup|IsHiddenInUI|OnlyAllowMembersViewMembership|AllowMembersEditMembership|AllowRequestToJoinLeave|AutoAcceptRequestToJoinLeave|RequestToJoinLeaveEmailSetting|OwnerTitle|OwnerCount| AssignedPermissions|AssignedPermissionCount|HasDirectPermissions|ComplianceStatus|ReviewRequired|       Snapsho

In [10]:
# # Fucntion for Calibrating source column definations to target column schema
from pyspark.sql.functions import col
from pyspark.sql.types import *

source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]

def align_to_target_schema_only(source_df, target_df, source_columns, target_columns):
    """  Align to match ONLY target schema (lose extra source columns)  """
    
    # Get target schema for type casting
    target_schema = {field.name: field.dataType for field in target_df.schema.fields}
    
    # Start with source DataFrame
    aligned_df = source_df.select(source_columns)
    target_df = target_df.select(target_columns)
    mapped_target_cols = set()

    # # STEP 1: Rename mapped columns (A→X, B→Y, C→Z)
    print(f"\n🔄 STEP 1: Renaming {len(source_columns)} mapped columns...")
    for src_col, tgt_col in zip(source_columns, target_columns):
        if src_col in aligned_df.columns:
            if src_col != tgt_col:  # Only rename if different
                aligned_df = aligned_df.withColumnRenamed(src_col, tgt_col)
                # print(f"  📝 {src_col} → {tgt_col}")
            # else:
                # print(f"  ✅ {src_col} (already correct name)")
            mapped_target_cols.add(tgt_col)
       # else:
            # print(f"  ⚠️ Source column '{src_col}' not found in source DF")
    
   # print(f"   After renaming: {aligned_df.columns}")
    
    # # STEP 2: Cast mapped columns to target types
    print(f"\n🔧 STEP 2: Casting {len(mapped_target_cols)} mapped columns...")
    for tgt_col in mapped_target_cols:
        if tgt_col in aligned_df.columns:
            target_type = target_schema.get(tgt_col, StringType())
            aligned_df = aligned_df.withColumn(tgt_col, col(tgt_col).cast(target_type))
           # print(f"  🔧 {tgt_col} cast to {target_type}")
   
    return aligned_df


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 12, Finished, Available, Finished)

In [11]:
# # Updating / triming source and target dataframes to required columns & renaming source columns to target column mapping 

source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]

# upd_source_df = source_df.filter(col("PrincipalType") == "User").select(source_cols)
upd_source_df = source_df.select(source_cols)
upd_target_df = target_df.select(target_cols)
print(f"Source Schema: {upd_source_df.printSchema()}")
print(f"Target Schema: {upd_target_df.printSchema()}")
upd_source_df = align_to_target_schema_only(upd_source_df, upd_target_df, source_cols, target_cols)

# print(f"Source schema  beefore:")
# print(f"{source_df.printSchema()}")
# print(f"Source post remapping:")
# print(f"{upd_source_df.printSchema()}")


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 13, Finished, Available, Finished)

root
 |-- GroupKey: string (nullable = true)
 |-- GroupId: integer (nullable = true)
 |-- LoginName: string (nullable = true)
 |-- PrincipalType: string (nullable = true)
 |-- OwnerLoginName: string (nullable = true)
 |-- OwnerEmail: string (nullable = true)
 |-- MemberCount: integer (nullable = true)
 |-- ComplianceStatus: string (nullable = true)

Source Schema: None
root
 |-- GroupKey: string (nullable = true)
 |-- GroupID: integer (nullable = true)
 |-- GroupName: string (nullable = true)
 |-- GroupType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- OwnerEmailID: string (nullable = true)
 |-- MemberCount: integer (nullable = true)
 |-- GroupStatus: string (nullable = true)

Target Schema: None

🔄 STEP 1: Renaming 8 mapped columns...

🔧 STEP 2: Casting 8 mapped columns...


In [12]:
from pyspark.sql.functions import col, current_timestamp

#     Detect INSERT records (rows in source not in target)
#     Args:
#         source_df: Source DataFrame (new data) 
#         target_df: Target DataFrame (existing data) 
#         sourcekey: Key column name in source DataFrame
#         targetkey: Key column name in target DataFrame 
#     Returns:
#         DataFrame with records to INSERT (from source)
# source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
# target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]

def detect_records2_inserts(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (source.key = target.key)
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Find records in source not in target
    inserts_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "left_anti")
        .withColumn("ModifiedDate", current_timestamp())
    )
    
    insert_count = inserts_df.count()
    print(f"🆕 INSERT: {insert_count} records found")
    
    return inserts_df.select(sourcekey)


    # Detect UPDATE records (rows exist in both but have different values)
    # Args:
    #     source_df: Source DataFrame (new data)
    #     target_df: Target DataFrame (existing data)
    #     sourcekey: Key column name in source DataFrame  
    #     targetkey: Key column name in target DataFrame
    # Returns:
    #     DataFrame with records to UPDATE (from source with new values)
    
def detect_records2updates(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Get non-key columns for comparison
    key_column = sourcekey
    non_key_cols = [c for c in source_df.columns if c != key_column]
    
    # Build dynamic "any column differs" filter
    diff_condition = None
    for c in non_key_cols:
        cond = col(f"s.{c}") != col(f"t.{c}")
        diff_condition = cond if diff_condition is None else (diff_condition | cond)
    
    # Find records that exist in both but have differences
    updates_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "inner")
        .filter(diff_condition)  # Only records with differences
        .select(
            col("s.*"),  # Take updated values from source
            current_timestamp().alias("ModifiedDate")
        )
    )
    
    update_count = updates_df.count()
    print(f"✏️ UPDATE: {update_count} records found")
    
    return updates_df.select(sourcekey)


    # Detect DELETE records (rows in target not in source)    
    # Args:
    #     source_df: Source DataFrame (new data)
    #     target_df: Target DataFrame (existing data)
    #     sourcekey: Key column name in source DataFrame
    #     targetkey: Key column name in target DataFrame
    # Returns:
    #     DataFrame with records to DELETE (from target)
    
def detect_records2_deletes(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (target.key = source.key) 
    join_condition = col(f"t.{targetkey}") == col(f"s.{sourcekey}")
    
    # Find records in target not in source
    deletes_df = (
        target_df.alias("t")
        .join(source_df.alias("s"), join_condition, "left_anti")
        .withColumn("LastModified", current_timestamp())
    )
    
    delete_count = deletes_df.count()
    print(f"🗑️ DELETE: {delete_count} records found")
    
    return deletes_df.select(targetkey)


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 14, Finished, Available, Finished)

In [13]:
# source_df.show(2)
# source_df.select("User_GUIDPK").show(2, truncate=False)

print(upd_source_df.count())
print(upd_target_df.count())

upd_source_df.show(2)
upd_target_df.printSchema()


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 15, Finished, Available, Finished)

212
212
+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+
|            GroupKey|GroupID|           GroupName|      GroupType|               Owner|OwnerEmailID|MemberCount|GroupStatus|
+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+
|72ed2ae4e3111585b...|      3|SMB Reporting Owners|SharePointGroup|SMB Reporting Owners|        NULL|          2|     Active|
|c2b4913efd5702387...|      4|SMB Reporting Vis...|SharePointGroup|SMB Reporting Owners|        NULL|          0|     Active|
+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+
only showing top 2 rows

root
 |-- GroupKey: string (nullable = true)
 |-- GroupID: integer (nullable = true)
 |-- GroupName: string (nullable = true)
 |-- GroupType: string (nullable = true)
 |-- Owner: string (nullable = true)
 |-- Ow

In [14]:
# # Code to check new records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *

# # Method 1a: Create single new record using Row
# upd_source_df.show(50)
new_record = spark.createDataFrame([
    Row(
            GroupKey = "4321865570610bab448aa00a7ff9359b1c3638f3fe6b87cb85baef5e279b8935",
            GroupID = 9,
            GroupName = "SMB Owners",
            GroupType = "PTGroup",
            Owner = "SMB Reporting",
            OwnerEmailID = "NULL",
            MemberCount = 6,
            GroupStatus = "Normal"
    )
])

upd_source_df = upd_source_df.union(new_record)

new_result_df = detect_records2_inserts(upd_source_df, upd_target_df,"GroupKey","GroupKey")
# print(f"target before:{target_df.count()}")

# Step 1: Join source with new_df to get complete records for new keys
# source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
# target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]
# Create column mapping (source -> target)
column_mapping = dict(zip(target_cols, target_cols))
new_records_df = (
    new_result_df.alias("n")
    .join(upd_source_df.alias("s"), col("n.GroupKey") == col("s.GroupKey"), "inner")
    .select(*[col(f"s.{source_cols}").alias(target_cols) for source_cols, target_cols in column_mapping.items()])
)

new_records_df.show()
# print(f"target before:{new_records.count()}")


# # # Union with target  
# updated_target = upd_target_df.union(new_complete_records)
# print(f"updated_target before:{updated_target.count()}")
# updated_target.filter(col("UserKey") == "26a5aa91b35f08966").show()

# # Get target columns
# target_columns = target_df.columns

# # Add missing columns with NULL and reorder to match target
# aligned_new_records = new_complete_records.select(*[
#     col(c) if c in new_complete_records.columns else lit(None).alias(c) 
#     for c in target_columns
# ])



StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 16, Finished, Available, Finished)

🆕 INSERT: 1 records found
+--------------------+-------+----------+---------+-------------+------------+-----------+-----------+
|            GroupKey|GroupID| GroupName|GroupType|        Owner|OwnerEmailID|MemberCount|GroupStatus|
+--------------------+-------+----------+---------+-------------+------------+-----------+-----------+
|4321865570610bab4...|      9|SMB Owners|  PTGroup|SMB Reporting|        NULL|          6|     Normal|
+--------------------+-------+----------+---------+-------------+------------+-----------+-----------+



In [15]:
# # Code to check removed record from source and remove from target table dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *
# source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
# target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]

# Delete Record in users 
# upd_source_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show()
# # Remove specific user by UserPrincipalName
upd_source_df = upd_source_df.filter(col("GroupID") != 5)
# upd_source_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show()

# # Perform removal of deleted records - in-memory merge
delete_result_df = detect_records2_deletes(upd_source_df, upd_target_df,"GroupKey","GroupKey")
delete_result_df.show(3)

# upd_target_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show(25)
# upd_target_df= upd_target_df.join(delete_result_df.select("UserKey"), "UserKey", "left_anti")
# upd_target_df.filter(col("UserPrincipalName") == "sureshgupta@peopletech.com").show(25)

print(f"Source deleted record has been locted")


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 17, Finished, Available, Finished)

🗑️ DELETE: 1 records found
+--------------------+
|            GroupKey|
+--------------------+
|432f4fab24632d5cf...|
+--------------------+

Source deleted record has been locted


In [16]:
# # Code to check updated records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import * 
# import lit, current_timestamp, col,when, upper, trim
from pyspark.sql.types import *
# source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
# target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]

# # Update Department for all users in "Software Engineering" 
upd_source_df.filter(col("GroupKey") == "72ed2ae4e3111585bc4c57854423ac94b5f8f34d6410af3af2ebbcde59410e44").show()

upd_source_df = upd_source_df.withColumn(
    "GroupType",
    when(col("GroupKey") == "72ed2ae4e3111585bc4c57854423ac94b5f8f34d6410af3af2ebbcde59410e44", "PtgGroup@hyd")
    .otherwise(col("GroupType"))
)

upd_source_df.filter(col("GroupKey") == "72ed2ae4e3111585bc4c57854423ac94b5f8f34d6410af3af2ebbcde59410e44").show()

# source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
# target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]
source_cols = ["GroupKey","GroupId","LoginName","PrincipalType"]
target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType"]

### showing 
upd_result_df = detect_records2updates(upd_source_df, upd_target_df,"GroupKey","GroupKey")
upd_result_df.show()



StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 18, Finished, Available, Finished)

+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+
|            GroupKey|GroupID|           GroupName|      GroupType|               Owner|OwnerEmailID|MemberCount|GroupStatus|
+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+
|72ed2ae4e3111585b...|      3|SMB Reporting Owners|SharePointGroup|SMB Reporting Owners|        NULL|          2|     Active|
+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+

+--------------------+-------+--------------------+------------+--------------------+------------+-----------+-----------+
|            GroupKey|GroupID|           GroupName|   GroupType|               Owner|OwnerEmailID|MemberCount|GroupStatus|
+--------------------+-------+--------------------+------------+--------------------+------------+-----------+-----------+


In [17]:
# # # Code to update target dataframe with updated / inserrted / deleted records *************8

# new_records_df.show()
# upd_result_df.show()
final_result_Key = (
    new_records_df.select(col("GroupKey").alias("GroupKey"))
    .union(upd_result_df.select(col("GroupKey").alias("GroupKey")))
)
final_result_Key.show()

# Get updated records with specified columns only
source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]
##GroupMailID	ParentGroup	Scope	ExpirationDate	CreatedBy	CreatedDate	ModifiedBy	ModifiedDate	SnapshotDate

updated_add_records = (
    final_result_Key.alias("n")
    .join(source_df.alias("s"), col("n.GroupKey") == col("s.GroupKey") , "inner")
    .select(*[f"s.{col}" for col in source_cols])
)
# updated_add_records.show()

# # # #  # # updated_add_records = align_to_target_schema_only(updated_add_records, updated_delrecords, source_cols, target_cols)

updated_add_records = updated_add_records \
    .withColumn("ParentGroup", lit(None).cast(StringType())) \
    .withColumn("GroupMailID", lit(None).cast(StringType())) \
    .withColumn("ExpirationDate", lit(None).cast(StringType())) \
    .withColumn("Scope", lit(None).cast(StringType())) \
    .withColumn("CreatedBy", lit("System").cast(StringType())) \
    .withColumn("CreatedDate", current_date()) \
    .withColumn("ModifiedBy", lit("System").cast(StringType())) \
    .withColumn("ModifiedDate", current_date()) \
    .withColumn("IsDeleted", lit(0)) \
    .withColumn("SnapshotDate", current_date()) 
    
updated_add_records.show(2)


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 19, Finished, Available, Finished)

+--------------------+
|            GroupKey|
+--------------------+
|4321865570610bab4...|
|72ed2ae4e3111585b...|
+--------------------+

+--------------------+-------+--------------------+---------------+--------------------+----------+-----------+----------------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|            GroupKey|GroupId|           LoginName|  PrincipalType|      OwnerLoginName|OwnerEmail|MemberCount|ComplianceStatus|ParentGroup|GroupMailID|ExpirationDate|Scope|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|SnapshotDate|
+--------------------+-------+--------------------+---------------+--------------------+----------+-----------+----------------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|72ed2ae4e3111585b...|      3|SMB Reporting Owners|SharePointGroup|SMB Reporting Owners|      NULL|          2|          Active|    

In [18]:
updated_add_records.show(1)

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 20, Finished, Available, Finished)

+--------------------+-------+--------------------+---------------+--------------------+----------+-----------+----------------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|            GroupKey|GroupId|           LoginName|  PrincipalType|      OwnerLoginName|OwnerEmail|MemberCount|ComplianceStatus|ParentGroup|GroupMailID|ExpirationDate|Scope|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|SnapshotDate|
+--------------------+-------+--------------------+---------------+--------------------+----------+-----------+----------------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|72ed2ae4e3111585b...|      3|SMB Reporting Owners|SharePointGroup|SMB Reporting Owners|      NULL|          2|          Active|       NULL|       NULL|          NULL| NULL|   System| 2025-10-07|    System|  2025-10-07|        0|  2025-10-07|
+--------------------+------

In [19]:
# # code to creeate delete recor dataframe and update isdeleted to 1 and other system values.
source_cols = ["GroupKey","GroupId","LoginName","PrincipalType","OwnerLoginName","OwnerEmail","MemberCount","ComplianceStatus"]
target_cols = ["GroupKey", "GroupID", "GroupName", "GroupType", "Owner", "OwnerEmailID", "MemberCount", "GroupStatus"]

updated_delrecords = (
    delete_result_df.alias("n")
    .join(upd_target_df.alias("s"), col("n.GroupKey") == col("s.GroupKey") , "inner")
    .select(*[f"s.{col}" for col in target_cols])
)

updated_delrecords = updated_delrecords \
    .withColumn("ParentGroup", lit(None).cast(StringType())) \
    .withColumn("GroupMailID", lit(None).cast(StringType())) \
    .withColumn("ExpirationDate", lit(None).cast(StringType())) \
    .withColumn("Scope", lit(None).cast(StringType())) \
    .withColumn("CreatedBy", lit("System").cast(StringType())) \
    .withColumn("CreatedDate", current_date()) \
    .withColumn("ModifiedBy", lit("System").cast(StringType())) \
    .withColumn("ModifiedDate", current_date()) \
    .withColumn("IsDeleted", lit(1)) \
    .withColumn("SnapshotDate", current_date()) 

# updated_delrecords.show()

final_records = (
     updated_delrecords
    .union(updated_add_records)
)
final_records.show(2)

target_df.show(1)

final_records = final_records.select(*target_df.columns)
final_records.show(5)

# # Remove records from target that exist in updadddeeel, then add updated records
final_target_rec = (
    target_df.join(final_records.select("GroupKey"), "GroupKey", "left_anti")  # Remove existing
    .union(final_records)  # Add updated records
)
# final_target_rec.show(5)


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 21, Finished, Available, Finished)

+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|            GroupKey|GroupID|           GroupName|      GroupType|               Owner|OwnerEmailID|MemberCount|GroupStatus|ParentGroup|GroupMailID|ExpirationDate|Scope|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|SnapshotDate|
+--------------------+-------+--------------------+---------------+--------------------+------------+-----------+-----------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|432f4fab24632d5cf...|      5|SMB Reporting Mem...|SharePointGroup|SMB Reporting Owners|        NULL|          1|     Active|       NULL|       NULL|          NULL| NULL|   System| 2025-10-07|    System|  2025-10-07|        1|  2025-10-07|
|72ed2ae4e3111585b...|      3|SMB Report

In [20]:
updated_add_records.show(1)
updated_delrecords.show(1)
# final_records.show(2)
# final_target_rec.show(2)

StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 22, Finished, Available, Finished)

+--------------------+-------+--------------------+---------------+--------------------+----------+-----------+----------------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|            GroupKey|GroupId|           LoginName|  PrincipalType|      OwnerLoginName|OwnerEmail|MemberCount|ComplianceStatus|ParentGroup|GroupMailID|ExpirationDate|Scope|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|SnapshotDate|
+--------------------+-------+--------------------+---------------+--------------------+----------+-----------+----------------+-----------+-----------+--------------+-----+---------+-----------+----------+------------+---------+------------+
|72ed2ae4e3111585b...|      3|SMB Reporting Owners|SharePointGroup|SMB Reporting Owners|      NULL|          2|          Active|       NULL|       NULL|          NULL| NULL|   System| 2025-10-07|    System|  2025-10-07|        0|  2025-10-07|
+--------------------+------

In [21]:
 # ===== WRITE PROCESS =====
try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_path}")

    full_target_path_new = full_target_path
    #####.rsplit('.', 1)[0]}_{datetime.now():%Y%m%d}.parquet"

    final_target_rec.write.mode("overwrite").option("compression", "snappy") \
    .parquet(f"{full_target_path_new}")

    # final_target.write.mode("overwrite").option("compression", "snappy") \
    # .parquet(f"{full_target_path.rsplit('.',1)[0]}_{datetime.now():%Y%m%d}.parquet")
    print(f"Successfully written to: {full_target_path_new}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_path_new)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 23, Finished, Available, Finished)

Writing to Gold layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Group.parquet
Successfully written to: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Group.parquet
Verification - Target record count: 212
Process completed!


In [22]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = source_df.count()
    rows_written = final_target_rec.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, cfe7f6a6-dee0-4e38-a225-6f73fc2fcd44, 24, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_Sil2Gld_DimGroup V2...
✅ ntk_Sil2Gld_DimGroup V2 completed successfully (233s)
🎉 ntk_Sil2Gld_DimGroup V2 pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 212 → 212
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_Sil2Gld_DimGroup V2:
+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_Sil2Gld_DimGroup V2 logging completed!
